# 00 — Setup & overview

**Foundry feature:** orientation — no Foundry call yet, just getting the environment and the case
study straight before you touch the portal.

## What is Microsoft Foundry Agent Service's prompt-agent optimizer?

Microsoft Foundry Agent Service lets you define a **prompt agent** — a system prompt
(`instructions.md`), an attached model, and a set of tools (function tools and/or MCP servers) — and
run it against real traffic. Its **Optimize** feature (preview, portal-only as of this writing) takes
a prompt agent plus a labeled dataset of `(query, ground_truth)` pairs and searches for a rewritten
instruction set that scores better against an LLM-judge evaluation, without you hand-editing the
prompt yourself.

That combination — a hosted, versioned, closed-preview optimizer you drive from a wizard — is exactly
what this notebook series shows you how to use *and* how to evaluate rigorously, because "the wizard
reported a higher score" is not, by itself, evidence that the optimized candidate is safe to deploy.

## The case study

Every notebook in this series drives the **same case study agent**: `01-travel-approval-strict`,
a "good baseline" travel-expense-approval prompt agent from the
[`prompt-agent-optimizer-baselines`](../prompt-agent-optimizer-baselines/) pack shipped in this repo.

It's a deliberate choice, not the default agent in the pack's numbering:

- It's a **control** agent (see the pack's `README.md` and `docs/agent-evaluation-guide.md`) — its
  baseline already follows instruction best practice, so you can watch Foundry's optimizer either
  respect a good prompt or damage it, without a second variable (a badly-written baseline) muddying
  what you're seeing.
- It has **3 function tools and no MCP server**, which is enough to show tool definitions and
  tool-call gating without the extra moving part of an MCP retrieval trigger (that's what the pack's
  `04-hr-policy-mcp` / `07-incident-response-mcp` agents are for, if you want a follow-up exercise).
- It has clear **numeric policy** (approval thresholds, lodging caps, a 6-hour business-class rule),
  so "did the optimizer preserve the policy or quietly reword it into something softer" is something
  you can check by eye, not just by score.

Swap `AGENT_ID` in notebook 00's setup cell for any of the pack's other nine agents to re-run this
whole series against a different case study (see `../prompt-agent-optimizer-baselines/docs/agent-evaluation-guide.md`
for what each one is designed to expose).

## Notebook roadmap

This is the plan for the whole series: ten notebooks, each demonstrating one part of the Foundry
prompt-agent optimizer workflow, in the order you'd actually run them.

| # | Notebook | Foundry feature it demonstrates | Mode |
|---|---|---|---|
| 00 | `00_setup_and_overview.ipynb` | Environment setup, the case-study agent, the notebook roadmap | CLI |
| 01 | `01_agent_anatomy.ipynb` | Anatomy of a Foundry **prompt agent** (instructions, tools, dataset, eval contract) | CODE |
| 02 | `02_create_agent_in_foundry_portal.ipynb` | Creating an agent in **Foundry Agent Service** | Portal |
| 03 | `03_prepare_and_upload_dataset.ipynb` | Preparing an **Optimize wizard** dataset upload | CLI + Portal |
| 04 | `04_run_optimize_wizard.ipynb` | Running the **Optimize wizard**: model selection, the run, exporting a candidate | Portal + CODE |
| 05 | `05_validate_candidate_contract.ipynb` | Deterministic contract gating of an optimized candidate | CLI |
| 06 | `06_resolve_semantic_rules_with_judge.ipynb` | **LLM-judge** resolution of semantic rules, self-preference bias check | CLI |
| 07 | `07_evaluate_optimized_candidate.ipynb` | Held-out evaluation, replicate seeds, when a delta is real | CLI |
| 08 | `08_compare_to_open_baseline.ipynb` | Foundry vs. the open **DSPy MIPROv2** baseline, cross-run similarity | CLI (real run optional) |
| 09 | `09_assemble_final_report.ipynb` | Rolling every artifact into one promotion decision | CODE |

## Prerequisites

| Need | Required for |
|---|---|
| A clone of this repo, notebook kernel running from `notebooks/` | All notebooks |
| Python 3.10+ | All notebooks |
| Foundry Agent Service preview access on your subscription | Notebooks 02, 04 (portal steps) |
| An API key for a judge model (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, ...), only if you want real
  judge calls instead of the free stub judge | Notebook 06 (optional) |
| `pip install -r ../prompt-agent-optimizer-baselines/_baselines/dspy_mipro/requirements.txt`, only
  if you want to actually run the open baseline instead of reading how | Notebook 08 (optional) |

Nothing below this cell spends money or needs portal access — every **CLI**-labeled cell in this
series runs offline against the files already checked into this repo. **Portal**-labeled steps are
manual instructions you follow in the Foundry UI; **CODE** cells run standalone Python.

## Environment check

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## Sanity-check the pack loads correctly

Before relying on any of this pack's data, confirm the case study's dataset split is intact: 20
optimize rows, 10 holdout rows, 3 tools, no MCP. This mirrors Step 1 of
`../prompt-agent-optimizer-baselines/docs/experiment-runbook.md`.

In [ ]:
optimize_rows = [json.loads(l) for l in (AGENT_DIR / "dataset" / "optimize.jsonl").read_text().splitlines() if l.strip()]
holdout_rows = [json.loads(l) for l in (AGENT_DIR / "dataset" / "holdout.jsonl").read_text().splitlines() if l.strip()]
tools = json.loads((AGENT_DIR / "tools.json").read_text())

print(f"optimize.jsonl : {len(optimize_rows)} rows")
print(f"holdout.jsonl  : {len(holdout_rows)} rows")
print(f"tools.json     : {len(tools)} function tools -> {[t['function']['name'] for t in tools]}")

## (Optional) set up the open DSPy baseline environment

Skip this until notebook 08 if you only care about the Foundry-portal half of the story. This is
Step 1 of the experiment runbook — a one-time environment setup for the open, reproducible
comparison baseline this repo ships alongside the Foundry track.

In [ ]:
# CLI (real setup, no API key/cost yet) -- uncomment to run
# run(["python3", "-m", "venv", ".venv"], cwd=PACK_ROOT / "_baselines" / "dspy_mipro")
# run(["./.venv/bin/pip", "install", "-r", "requirements.txt"], cwd=PACK_ROOT / "_baselines" / "dspy_mipro")
print("See notebook 08 for the DSPy baseline walkthrough.")

## Next

Continue to **`01_agent_anatomy.ipynb`** to look at exactly what makes up the case-study prompt
agent before creating it in Foundry.